In [1]:
pip install numpy pandas scikit-learn xgboost matplotlib seaborn

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier

# Create directory structure for downloads/outputs
OUTPUT_DIR = "loan_risk_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[*] Directories initialized. Outputs will be saved to: {OUTPUT_DIR}/")

# -------------------------------------------------------------------------
# STEP 1: RESILIENT DATASET LOADING (PROGRAMMATIC CREDIT PROFILE MATCHING)
# -------------------------------------------------------------------------
CSV_PATH = os.path.join(OUTPUT_DIR, "home_credit_sample.csv")

if not os.path.exists(CSV_PATH):
    print("[*] Generating statistical Home Credit risk dataset...")
    np.random.seed(42)
    n_samples = 5000

    # Simulating standard financial risk variables
    income = np.random.lognormal(mean=11.0, sigma=0.5, size=n_samples) # Average ~65k
    credit_amount = income * np.random.uniform(1.5, 4.0, size=n_samples)
    annuity = credit_amount * np.random.uniform(0.05, 0.12, size=n_samples)
    employment_length = np.random.exponential(scale=5, size=n_samples)

    # Contract types and housing situations
    contract_type = np.random.choice(['Cash loans', 'Revolving loans'], size=n_samples, p=[0.85, 0.15])
    housing_type = np.random.choice(['House / apartment', 'With parents', 'Rented apartment'], size=n_samples, p=[0.88, 0.08, 0.04])

    # Creating a ground truth target mapping where structural risk drives defaults
    # High credit-to-income ratio + short employment = higher default probability
    risk_score = (credit_amount / income) * 1.5 - (employment_length * 0.2) + np.random.normal(0, 1.0, size=n_samples)
    # Target distribution approx 8-9% default rate matching the official Home Credit matrix
    target = (risk_score > np.percentile(risk_score, 91)).astype(int)

    df = pd.DataFrame({
        'SK_ID_CURR': range(100001, 100001 + n_samples),
        'TARGET': target,
        'NAME_CONTRACT_TYPE': contract_type,
        'AMT_INCOME_TOTAL': income,
        'AMT_CREDIT': credit_amount,
        'AMT_ANNUITY': annuity,
        'NAME_HOUSING_TYPE': housing_type,
        'DAYS_EMPLOYED': employment_length * 365 # Convert to days format
    })

    df.to_csv(CSV_PATH, index=False)
    print("[+] Risk dataset initialized successfully.")
else:
    df = pd.read_csv(CSV_PATH)

# -------------------------------------------------------------------------
# STEP 2: CLEANING, PREPROCESSING & ENCODING
# -------------------------------------------------------------------------
print("[*] Preprocessing risk attributes and handling category data...")

# Label encode categorical columns
cat_cols = ['NAME_CONTRACT_TYPE', 'NAME_HOUSING_TYPE']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Feature Engineering: Income-to-Credit Ratios
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# Split features and target
X = df.drop(columns=['SK_ID_CURR', 'TARGET'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalize scaling for models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------------
# STEP 3: TRAIN BINARY CLASSIFIER MODEL
# -------------------------------------------------------------------------
print("[*] Training advanced XGBoost Risk Classifier...")
# Scale position weight balances the minority default class target distribution
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)
model = XGBClassifier(n_estimators=150, learning_rate=0.05, scale_pos_weight=scale_weight, random_state=42)
model.fit(X_train_scaled, y_train)

# Get default risk probabilities
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# -------------------------------------------------------------------------
# STEP 4: BUSINESS COST OPTIMIZATION MATRIX
# -------------------------------------------------------------------------
print("[*] Applying financial cost optimization sequence...")

# Define business variables
COST_FN = 5000  # Cost of a False Negative (Missing a defaulter = unpaid principal loss)
COST_FP = 500   # Cost of a False Positive (Rejecting a good customer = lost interest revenue)

thresholds = np.linspace(0, 1, 101)
total_costs = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)

    # Unpack confusion metrics securely
    # cm layout: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = cm.ravel()

    # Calculate operational cost value
    business_cost = (fn * COST_FN) + (fp * COST_FP)
    total_costs.append(business_cost)

# Locate optimal business threshold minimizing financial losses
best_idx = np.argmin(total_costs)
optimal_threshold = thresholds[best_idx]
min_cost = total_costs[best_idx]
default_cost = confusion_matrix(y_test, (y_prob >= 0.5).astype(int)).ravel()[2] * COST_FN + confusion_matrix(y_test, (y_prob >= 0.5).astype(int)).ravel()[1] * COST_FP

print(f"[+] Optimal Financial Decision Threshold located at: {optimal_threshold:.2f}")
print(f"[+] Total operational cost reduced to: ${min_cost:,.2f} vs Standard 0.5 threshold cost: ${default_cost:,.2f}")

# -------------------------------------------------------------------------
# STEP 5: SAVE VISUALIZATIONS AND SUMMARY REPORT
# -------------------------------------------------------------------------
print("[*] Generating evaluative cost optimization plots...")

# 1. Cost vs Threshold Curve
plt.figure(figsize=(9, 5))
plt.plot(thresholds, total_costs, color='red', linewidth=2.5, label='Total Operational Cost ($)')
plt.axvline(optimal_threshold, color='green', linestyle='--', label=f'Optimal Threshold ({optimal_threshold:.2f})')
plt.axvline(0.5, color='gray', linestyle=':', label='Standard ML Threshold (0.50)')
plt.title('Business Cost Minimization Curve Across Threshold Spectrum', fontsize=13)
plt.xlabel('Probability Decision Threshold')
plt.ylabel('Total Bank Financial Loss ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_business_cost_curve.png'))
plt.close()

# 2. Risk ROC Curve Visual
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Risk Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_loan_risk_roc_curve.png'))
plt.close()

# Save complete metrics report
y_pred_optimal = (y_prob >= optimal_threshold).astype(int)
with open(os.path.join(OUTPUT_DIR, "financial_optimization_report.txt"), "w") as f:
    f.write("===================================================\n")
    f.write("    LOAN RISK & FINANCIAL COST OPTIMIZATION REPORT \n")
    f.write("===================================================\n\n")
    f.write(f"Optimal Threshold: {optimal_threshold:.2f}\n")
    f.write(f"Minimized Portfolio Financial Loss: ${min_cost:,.2f}\n")
    f.write(f"Standard Model Threshold Loss (0.50): ${default_cost:,.2f}\n")
    f.write(f"Total Bank Capital Saved: ${default_cost - min_cost:,.2f}\n\n")
    f.write("=== Classification Matrix at Optimal Threshold ===\n")
    f.write(classification_report(y_test, y_pred_optimal))

print(f"\n[Done] Pipeline execution finalized completely! Inspect '{OUTPUT_DIR}' files.")

[*] Directories initialized. Outputs will be saved to: loan_risk_outputs/
[*] Generating statistical Home Credit risk dataset...
[+] Risk dataset initialized successfully.
[*] Preprocessing risk attributes and handling category data...
[*] Training advanced XGBoost Risk Classifier...
[*] Applying financial cost optimization sequence...
[+] Optimal Financial Decision Threshold located at: 0.23
[+] Total operational cost reduced to: $162,500.00 vs Standard 0.5 threshold cost: $208,500.00
[*] Generating evaluative cost optimization plots...

[Done] Pipeline execution finalized completely! Inspect 'loan_risk_outputs' files.
